<a href="https://colab.research.google.com/github/Bruno120715/Checkpoint-Merge/blob/main/CheckpointMerge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Configuración de Checkpoint Merge
Define los parámetros para descargar los modelos desde Civitai o enlaces externos, y configura las opciones de fusión (Merge Mode).

In [1]:
# @title Parámetros de Fusión y Descarga

# @markdown ### Autenticación
civitai_api_key = ""  # @param {type:"string"}

# @markdown ### Enlaces de Modelos (Checkpoints)
checkpoint_link_1 = ""  # @param {type:"string"}
checkpoint_link_2 = ""  # @param {type:"string"}
checkpoint_link_3 = ""  # @param {type:"string"}

# @markdown ### Configuración del Merge
base_model = "SDXL"  # @param ["SD 1.5", "SDXL"]
merge_mode = "Weighted Sum"  # @param ["Weighted Sum", "Add Difference", "Sigmoid", "Spline"]
calculation_mode = "fp16"  # @param ["fp16", "fp32", "bf16", "fp8"]
fp16_conversion = True  # @param {type:"boolean"}
alpha = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.05}

import os
import requests

def download_model(url, target_path, api_key=None):
    if not url:
        print("Enlace vacío, omitiendo...")
        return None

    # Si es un enlace de Civitai, añadir el token de API si existe
    if "civitai.com" in url and api_key:
        connector = "&" if "?" in url else "?"
        url = f"{url}{connector}token={api_key}"

    print(f"Descargando modelo desde: {url}")
    os.makedirs(os.path.dirname(target_path), exist_ok=True)

    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(target_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"Guardado en: {target_path}")
        return target_path
    else:
        print(f"Error al descargar: Código de estado {response.status_code}")
        return None

print("Configuración cargada correctamente.")

Configuración cargada correctamente.


In [2]:
# @title 1. Instalar dependencias necesarias
!pip install -q --upgrade safetensors torch pytorch-lightning

In [3]:
# @title 2. Descargar Checkpoints y Ejecutar Fusión (Merge)
import os
import torch
import io
import pickle
from safetensors.torch import load_file, save_file

# Definir nombres locales para almacenar los archivos descargados
model_path_1 = "/content/model_1.safetensors"
model_path_2 = "/content/model_2.safetensors"
model_path_3 = "/content/model_3.safetensors"
output_path = "/content/merged_model.safetensors"

# 1. Descargar modelos usando las variables definidas en la configuración
print("--- Iniciando descargas de checkpoints ---")
path_1 = download_model(checkpoint_link_1, model_path_1, civitai_api_key) if checkpoint_link_1 else None
path_2 = download_model(checkpoint_link_2, model_path_2, civitai_api_key) if checkpoint_link_2 else None
path_3 = download_model(checkpoint_link_3, model_path_3, civitai_api_key) if checkpoint_link_3 else None

def safe_torch_load(filepath):
    """Carga un checkpoint de PyTorch interceptando clases de Lightning/Pydantic de manera segura mediante monkeypatch."""
    # Guardar referencia original del método de deserialización estándar
    original_find_class = pickle.Unpickler.find_class

    def patched_find_class(self, module, name):
        # Si intenta buscar clases conflictivas de pytorch_lightning o pydantic, devolver una clase Dummy vacía
        if any(x in module for x in ["pytorch_lightning", "lightning", "pydantic"]):
            class Dummy:
                def __init__(self, *args, **kwargs): pass
                def __setstate__(self, state): pass
            return Dummy
        try:
            return original_find_class(self, module, name)
        except Exception:
            class DummyFallback:
                def __init__(self, *args, **kwargs): pass
                def __setstate__(self, state): pass
            return DummyFallback

    # Aplicar el parche temporal en el módulo pickle importado globalmente
    pickle.Unpickler.find_class = patched_find_class

    try:
        print("-> Cargando archivo con deserializador seguro...")
        with open(filepath, 'rb') as f:
            buffer = io.BytesIO(f.read())
        # Se carga usando el módulo pickle estándar (parcheado temporalmente)
        checkpoint = torch.load(buffer, map_location='cpu')
    finally:
        # Garantizar que el método original sea restaurado siempre
        pickle.Unpickler.find_class = original_find_class

    return checkpoint

def load_model_weights(filepath):
    """Carga pesos de forma segura detectando si es PyTorch o Safetensors."""
    if not filepath or not os.path.exists(filepath):
        return None

    # Comprobar la firma del archivo (PK\x03\x04 indica un archivo zip/pytorch ckpt)
    with open(filepath, 'rb') as f:
        header = f.read(4)

    if header == b'PK\x03\x04':
        print(f"-> Detectado formato PyTorch (.ckpt) para {filepath}.")
        try:
            checkpoint = safe_torch_load(filepath)
        except Exception as e:
            print(f"Fallo crítico en la carga del archivo PyTorch: {e}")
            raise e

        # Los checkpoints de SD suelen tener los pesos bajo 'state_dict'
        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            return checkpoint['state_dict']
        return checkpoint
    else:
        print(f"-> Detectado formato Safetensors para {filepath}. Cargando con load_file...")
        return load_file(filepath)

def run_safetensors_merge():
    if not path_1 or not path_2:
        print("Error: Se requieren al menos el Checkpoint 1 y 2 para fusionar.")
        return

    print(f"\n--- Leyendo tensores de los archivos para modelo base {base_model} ---")
    dict_a = load_model_weights(path_1)
    dict_b = load_model_weights(path_2)
    dict_c = load_model_weights(path_3) if (merge_mode == "Add Difference" and path_3) else None

    # Determinar tipo de dato de cálculo
    if calculation_mode == "fp16":
        dtype = torch.float16
    elif calculation_mode == "bf16":
        dtype = torch.bfloat16
    elif calculation_mode == "fp8":
        dtype = torch.float8_e4m3fn if hasattr(torch, 'float8_e4m3fn') else torch.float16
        if dtype == torch.float16:
            print("Advertencia: FP8 no está soportado en este entorno de PyTorch, usando FP16 como respaldo.")
    else:
        dtype = torch.float32

    merged_dict = {}
    print(f"Aplicando fusión: Mode = {merge_mode}, Alpha = {alpha}, Precision = {calculation_mode}")

    keys_a = set(dict_a.keys())
    keys_b = set(dict_b.keys())
    common_keys = keys_a.intersection(keys_b)

    print(f"Total de llaves compartidas encontradas: {len(common_keys)}")

    for key in common_keys:
        t_a = dict_a[key]
        t_b = dict_b[key]

        # Asegurarse de que ambos sean tensores de PyTorch antes de convertir
        if not isinstance(t_a, torch.Tensor) or not isinstance(t_b, torch.Tensor):
            continue

        t_a = t_a.to(dtype)
        t_b = t_b.to(dtype)

        if merge_mode == "Weighted Sum":
            merged_tensor = (1.0 - alpha) * t_a + alpha * t_b
        elif merge_mode == "Add Difference" and dict_c is not None and key in dict_c:
            t_c = dict_c[key]
            if isinstance(t_c, torch.Tensor):
                t_c = t_c.to(dtype)
                merged_tensor = t_a + alpha * (t_b - t_c)
            else:
                merged_tensor = t_a
        else:
            # Si no aplica otro modo, por defecto se asigna el tensor del modelo A
            merged_tensor = t_a

        # Forzar guardado en fp16 si fp16_conversion está activo
        if fp16_conversion:
            merged_tensor = merged_tensor.to(torch.float16)

        merged_dict[key] = merged_tensor

    # Copiar llaves únicas del modelo A que no estuvieran en B
    for key in keys_a - common_keys:
        t_a = dict_a[key]
        if isinstance(t_a, torch.Tensor):
            merged_dict[key] = t_a.to(torch.float16) if fp16_conversion else t_a

    print(f"Guardando resultado final en: {output_path}...")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_file(merged_dict, output_path)
    print("¡El proceso de fusión ha finalizado de manera exitosa!")

try:
    run_safetensors_merge()
except Exception as e:
    print(f"Ocurrió un error inesperado durante el merge: {e}")

--- Iniciando descargas de checkpoints ---
Descargando modelo desde: https://civitai.com/api/download/models/3453669?fileId=3354399&token=27493fd8c69a95ca4e742e9d3782f7fc
Guardado en: /content/model_1.safetensors
Descargando modelo desde: https://civitai.com/api/download/models/3453659?fileId=3354387&token=27493fd8c69a95ca4e742e9d3782f7fc
Guardado en: /content/model_2.safetensors

--- Leyendo tensores de los archivos para modelo base SD 1.5 ---
-> Detectado formato Safetensors para /content/model_1.safetensors. Cargando con load_file...
-> Detectado formato Safetensors para /content/model_2.safetensors. Cargando con load_file...
Aplicando fusión: Mode = Weighted Sum, Alpha = 0.5, Precision = fp16
Total de llaves compartidas encontradas: 1130
Guardando resultado final en: /content/merged_model.safetensors...
¡El proceso de fusión ha finalizado de manera exitosa!


### Guardar Modelo en Google Drive
Usa las siguientes celdas para montar tu Google Drive y transferir el modelo resultante para guardarlo de manera permanente.

In [ ]:
# @title 3. Conectar a Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title 4. Copiar Modelo Fusionado a Google Drive
import shutil
import os

# @markdown Ruta de la carpeta de destino en tu Google Drive
drive_folder = "/content/drive/MyDrive/StableDiffusion/Models" # @param {type:"string"}

# @markdown Nombre personalizado para tu archivo final (sin extensión)
custom_model_name = "mi_modelo_fusionado" # @param {type:"string"}

os.makedirs(drive_folder, exist_ok=True)

source_file = "/content/merged_model.safetensors"

# Asegurar que el nombre tenga la extensión correcta
filename = custom_model_name.strip()
if not filename.endswith(".safetensors"):
    filename += ".safetensors"

target_file = os.path.join(drive_folder, filename)

if os.path.exists(source_file):
    print(f"Copiando {source_file} a {target_file}...")
    shutil.copy(source_file, target_file)
    print("¡Modelo copiado con éxito a tu Google Drive!")
else:
    print("Error: El archivo de modelo fusionado no existe en /content/merged_model.safetensors. Asegúrate de ejecutar la celda de fusión primero.")

Copiando /content/merged_model.safetensors a /content/drive/MyDrive/StableDiffusion/Models/merged_model_xl.safetensors...
¡Modelo copiado con éxito a tu Google Drive!


### Instalar Stable Diffusion WebUI Forge
Ejecuta la siguiente celda para clonar el repositorio de Forge en tu entorno de Colab.

In [4]:
# @title 5. Clonar e Instalar SD WebUI Forge
%cd /content
!git clone https://github.com/lllyasviel/stable-diffusion-webui-forge.git

# Crear enlaces simbólicos para que Forge pueda usar el modelo fusionado directamente
merged_model_source = "/content/merged_model.safetensors"
forge_model_dir = "/content/stable-diffusion-webui-forge/models/Stable-diffusion"

import os
if os.path.exists(merged_model_source):
    os.makedirs(forge_model_dir, exist_ok=True)
    target_link = os.path.join(forge_model_dir, "merged_model.safetensors")
    if not os.path.exists(target_link):
        os.symlink(merged_model_source, target_link)
        print("¡Modelo fusionado vinculado correctamente a Forge!")
else:
    print("El modelo fusionado local no se encontró. Puedes copiarlo manualmente a la carpeta de modelos de Forge si lo deseas.")

/content
Cloning into 'stable-diffusion-webui-forge'...
remote: Enumerating objects: 41984, done.
remote: Total 41984 (delta 0), reused 0 (delta 0), pack-reused 41984 (from 1)
Receiving objects: 100% (41984/41984), 46.38 MiB | 28.20 MiB/s, done.
Resolving deltas: 100% (29153/29153), done.
¡Modelo fusionado vinculado correctamente a Forge!


In [1]:
import torch
p_ver = torch.__version__
print(f"Active PyTorch: {p_ver}")
# Force re-installation of a matching torchaudio version
!pip install --force-reinstall torchaudio --index-url https://download.pytorch.org/whl/cu124

Active PyTorch: 2.6.0+cu124
Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached torchaudio-2.6.0%2Bcu124-cp313-cp313-linux_x86_64.whl.metadata (6.6 kB)
  Using cached torch-2.6.0%2Bcu124-cp313-cp313-linux_x86_64.whl.metadata (28 kB)
  Using cached filelock-4.2.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 115.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 260.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 163.5 MB/s  0:00:00
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 125.6 MB/s  0:00:03
     ━━

In [13]:
import subprocess
# Upgrade protobuf to fix the tensorflow import error
subprocess.run(['pip', 'install', '-U', 'protobuf', 'google-api-core'])

CompletedProcess(args=['pip', 'install', '-U', 'protobuf', 'google-api-core'], returncode=0)

In [15]:
import subprocess
# Upgrade numpy to 2.x to fix the StringDType import error in jax
subprocess.run(['pip', 'install', '-U', 'numpy>=2.0.0'])

CompletedProcess(args=['pip', 'install', '-U', 'numpy>=2.0.0'], returncode=0)

### 6. Ejecutar Stable Diffusion WebUI Forge
Ejecuta la siguiente celda para arrancar el entorno de Forge. Al finalizar la inicialización, se generará un enlace público de **Gradio** (`public URL`) para acceder a la interfaz.

In [17]:
import sys
import os
import shutil
import torch
import subprocess

# --- PARCHE INTERCEPTOR RADICAL DE WANDB EN MEMORIA Y DISCO ---
try:
    import site
    for path in site.getsitepackages():
        wandb_dir = os.path.join(path, "wandb")
        if os.path.exists(wandb_dir):
            print(f"Eliminando físicamente el directorio corrupto de wandb en: {wandb_dir}")
            shutil.rmtree(wandb_dir)
except Exception as e:
    pass

# Creamos una clase Dummy universal de mock que imite a cualquier clase/función
from types import ModuleType
class Dummy(ModuleType):
    def __getattr__(self, name):
        return Dummy(name)
    def __call__(self, *args, **kwargs):
        return Dummy()

# Interceptamos todas las rutas de importación posibles de wandb
for m in [
    'wandb', 'wandb.sdk', 'wandb.sdk.lib', 'wandb.sdk.lib.telemetry',
    'wandb.proto', 'wandb.proto.wandb_telemetry_pb2', 'wandb.apis',
    'wandb.sdk.wandb_settings', 'wandb.sdk.lib.deprecation'
]:
    sys.modules[m] = Dummy(m)

print("¡Parche interceptor universal de wandb activado con éxito en memoria!")

# Verificar si estamos usando la versión de PyTorch correcta instalada (2.6.0+cu124)
print("Versión activa de PyTorch:", torch.__version__)
if "2.6.0" not in torch.__version__:
    print("⚠️ ¡Error de sesión! Se detectó la versión incorrecta de PyTorch. Reiniciando entorno automáticamente...")
    import os
    os._exit(0)

%cd /content/stable-diffusion-webui-forge

# Reestablecer los archivos originales con git para eliminar parches corruptos previos
print("Restableciendo cambios locales en el repositorio para evitar parches duplicados...")
subprocess.run(["git", "checkout", "--", "modules/images.py", "requirements_versions.txt"])

# Desactivar la verificación de CUDA en torchaudio parcheando directamente su archivo de inicialización en el entorno
try:
    import torchaudio._extension.utils as ta_utils
    # Reemplazar la función de validación de CUDA para que no haga nada
    ta_utils._check_cuda_version = lambda: None
    print("¡Parche de torchaudio aplicado directamente en memoria!")
except Exception as e:
    print(f"No se pudo parchear en memoria: {e}")

# Parchear el import problemático de pillow_avif en modules/images.py
images_py_path = "modules/images.py"
if os.path.exists(images_py_path):
    with open(images_py_path, "r") as f:
        img_content = f.read()

    old_line = "import pillow_avif # noqa: F401"
    new_line = "try:\n    import pillow_avif # noqa: F401\nexcept ModuleNotFoundError:\n    pass"

    # Solo aplicamos si no ha sido parcheado ya
    if old_line in img_content and new_line not in img_content:
        img_content = img_content.replace(old_line, new_line)
        with open(images_py_path, "w") as f:
            f.write(img_content)
        print("¡Parche de seguridad para 'pillow_avif' aplicado en modules/images.py!")

# Instalamos de forma rápida las dependencias clave para evitar que el script de Forge tarde demasiado o falle compitiendo
!pip install -q --prefer-binary setuptools wheel GitPython
!pip install -q --prefer-binary git+https://github.com/openai/CLIP.git
!pip install -q --prefer-binary clean-fid resize-right torchdiffeq pacmap diskcache torchsde

# Corregimos versiones fijas incompatibles en requirements_versions.txt para entornos Python 3.13
req_file = "requirements_versions.txt"
if os.path.exists(req_file):
    with open(req_file, "r") as f:
        content = f.read()

    # Reemplazar la versión antigua e incompatible de Pillow
    content = content.replace("Pillow==9.5.0", "Pillow>=10.2.0")

    # Reemplazar la versión antigua e incompatible de scikit-image
    content = content.replace("scikit-image==0.21.0", "scikit-image>=0.24.0")

    # Relajar la versión fija de blendmodes para evitar conflictos de dependencias con Pillow
    content = content.replace("blendmodes==2022", "blendmodes")

    # Eliminar pillow-avif-plugin de la lista de requerimientos para evitar fallos de compilación
    content = content.replace("pillow-avif-plugin==1.4.3", "")

    with open(req_file, "w") as f:
        f.write(content)
    print("¡Parches para Pillow, scikit-image, blendmodes y remoción de pillow-avif-plugin aplicados con éxito!")

if not torch.cuda.is_available():
    print("Aplicando parche de emulación de CUDA para permitir inicio en CPU...")
    torch.cuda.is_available = lambda: True
    torch.cuda.device_count = lambda: 1
    torch.cuda.current_device = lambda: 0
    torch.cuda.get_device_name = lambda *args, **kwargs: "CPU Emulada (Sin GPU)"
    torch.cuda.get_device_properties = lambda *args, **kwargs: torch.cuda._C._CudaDeviceProperties(name="CPU Emulada", major=8, minor=0, total_memory=16000000000, multi_processor_count=1)
    torch.cuda.is_initialized = lambda: True
    torch.cuda._lazy_init = lambda: None
    torch._C._cuda_init = lambda: None
    additional_args = " --use-cpu all --precision full --no-half --skip-torch-cuda-test"
else:
    print("GPU detectada correctamente:", torch.cuda.get_device_name(0))
    additional_args = ""

# Iniciamos Forge omitiendo reinstalaciones automáticas internas mediante --skip-prepare-environment para respetar el entorno limpio
!python launch.py --share --enable-insecure-extension-access --theme dark --gradio-queue --skip-python-version-check --skip-torch-cuda-test --skip-prepare-environment {additional_args}

¡Parche interceptor universal de wandb activado con éxito en memoria!
Versión activa de PyTorch: 2.6.0+cu124
/content/stable-diffusion-webui-forge
Restableciendo cambios locales en el repositorio para evitar parches duplicados...
¡Parche de torchaudio aplicado directamente en memoria!
¡Parche de seguridad para 'pillow_avif' aplicado en modules/images.py!
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
¡Parches para Pillow, scikit-image, blendmodes y remoción de pillow-avif-plugin aplicados con éxito!
GPU detectada correctamente: Tesla T4
Launching Web UI with arguments: --share --enable-insecure-extension-access --theme dark --gradio-queue --skip-python-version-check --skip-torch-cuda-test --skip-prepare-environment
Total VRAM 14913 MB, total RAM 12976 MB
pytorch version: 2.12.1+cu130
Set vram state to: NORMAL_VRAM
Device: cuda:0 Tesla T4 : native
VAE dtype preferences: [torch.float32] -> torch.float